# Guide: LAB 03 — Delta DML & Time Travel

## Scenario

> *"A batch of updated customer records has arrived. You need to merge them into the existing table, then simulate a disaster (accidental DELETE) and recover data using Delta Lake's Time Travel. Finally, understand how VACUUM affects your ability to travel back in time."*



## Objectives

After completing this lab you will be able to:
- Use `MERGE INTO` for upsert operations (insert + update)
- Perform `UPDATE` and `DELETE` on Delta tables
- Inspect table history with `DESCRIBE HISTORY`
- Query previous versions with Time Travel (`VERSION AS OF`)
- Restore a table to a previous version with `RESTORE TABLE`
- Understand VACUUM and its impact on Time Travel



## Prerequisites

- Notebook attached to **Serverless** compute
- `00_setup` only — this lab does not require LAB 02 tables
- The Setup cells recreate `bronze.customers` from `customers.csv`


## Tasks Overview

Open the lab notebook **`lab_03_delta_dml.ipynb`** (in `notebooks/day1/lab/`) and complete the `# TODO` cells.

| Task | What to do | Key concept |
|------|-----------|-------------|
| **Task 1** | Examine the Update File | Load `customers_new.csv` into `df_updates`, compare counts |
| **Task 2** | MERGE INTO (Upsert) | Match on `customer_id`, UPDATE matched, INSERT new |
| **Task 3** | UPDATE Records | `UPDATE ... SET state = 'TX' WHERE city = 'Austin'` |
| **Task 4** | Accidental DELETE (run only, no code to write) | Provided `DELETE` removes every row |
| **Task 5** | DESCRIBE HISTORY | View all operations in table history |
| **Task 6** | Time Travel: Query the Previous Version | `SELECT * FROM table VERSION AS OF n` |
| **Task 7** | RESTORE the Table | `RESTORE TABLE ... TO VERSION AS OF n` |
| **Task 8** | VACUUM and Its Impact on Time Travel | Run VACUUM, observe Time Travel failure |


## Detailed Hints

### Task 1 — Examine the Update File
- Read `customers_new.csv` using `spark.read` with CSV format (`header`, `inferSchema`)
- Compare `df_updates.count()` (14 rows) with the base table count (10,000)

### Task 2 — MERGE INTO (Upsert)
- `v_updates` is registered for you in the cell before the TODO
- Match condition: `target.customer_id = source.customer_id`
- `WHEN MATCHED THEN UPDATE SET *`
- `WHEN NOT MATCHED THEN INSERT *`
- Expected: 10,006 rows after the MERGE

### Task 3 — UPDATE Records
- `SET state = 'TX'` and `WHERE city = 'Austin'`
- The Austin rows come from `customers_new.csv` (merged in Task 2) with wrong states — the validation checks they are all `TX` now

### Task 4 — Accidental DELETE (run only, no code to write)
- The provided `DELETE ... WHERE country IS NOT NULL` removes **every** row — no customer has a NULL country

### Task 5 — DESCRIBE HISTORY
- Command: `DESCRIBE HISTORY catalog.schema.table`
- Look for operations: WRITE / CREATE TABLE AS SELECT, MERGE, UPDATE, DELETE

### Task 6 — Time Travel: Query the Previous Version
- `version_before_delete` = the version of the DELETE minus 1 (the UPDATE)
- Use `VERSION AS OF {version_before_delete}` in SELECT

### Task 7 — RESTORE the Table
- `RESTORE TABLE ... TO VERSION AS OF {version_before_delete}`
- Use the same version as Task 6

### Task 8 — VACUUM and Its Impact on Time Travel
- The provided cell runs `OPTIMIZE` first: with deletion vectors, version 0's original files may still be referenced by the current table, so VACUUM alone would not remove them
- It also bypasses the retention safety check: Spark conf `spark.databricks.delta.retentionDurationCheck.enabled` on classic compute; on serverless (conf not available) the table property `delta.deletedFileRetentionDuration` is lowered instead
- `VACUUM table_name RETAIN 0 HOURS` (demo only, never in production!)
- After VACUUM, querying `VERSION AS OF 0` fails — wrap the query **and an action** (`.count()`) in `try/except` and set `time_travel_failed`

**Why the provided cell runs OPTIMIZE first.** With **deletion vectors** (default on new tables) MERGE/UPDATE mark rows as deleted instead of rewriting files, and RESTORE re-adds old files. Version 0's original data files can therefore still be referenced by the current version — VACUUM would keep them and `VERSION AS OF 0` would still work. OPTIMIZE rewrites the files so the old ones become unreferenced.

**Serverless.** `spark.databricks.delta.retentionDurationCheck.enabled` is not available on serverless compute. The provided cell falls back to lowering the table property `delta.deletedFileRetentionDuration`, which is the retention the VACUUM safety check compares against. `RETAIN 0 HOURS` is possible only because of this bypass (Spark conf on classic compute, table property on serverless).

**Which error you see.** After VACUUM, reading `VERSION AS OF 0` raises an error — typically a missing-file error (the data files are gone) or, on recent runtimes with a lowered `delta.deletedFileRetentionDuration`, an error saying time travel to that version is blocked.


## Summary

In this lab you:
- Used MERGE INTO for upsert (insert + update)
- Performed UPDATE and DELETE on Delta tables
- Inspected history with DESCRIBE HISTORY
- Queried previous versions with Time Travel
- Restored a table with RESTORE TABLE
- Ran VACUUM and observed its impact on Time Travel

> **Exam tip:** Time Travel uses the Delta transaction log. Data files for old versions are only removed by `VACUUM`. Default retention is **7 days**. After VACUUM, `DESCRIBE HISTORY` still shows metadata, but querying old versions fails because the underlying Parquet files are gone.

> **What's next:** In LAB 04 you will optimize Delta tables with OPTIMIZE, Z-ORDER, VACUUM, and Liquid Clustering.